In [ ]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle
import matplotlib.pyplot as plt
import time
import os
import copy
from scipy.stats import anderson
import statsmodels.api as sm
from datetime import datetime
from ordered_set import OrderedSet

In [ ]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_pickle("Data/df_data_2020_all_columns_08_11.pickle")

In [ ]:
with open("Data/country_to_currency_mapping.pickle", "rb") as f:
    country_to_currency_mapping = pickle.load(f)

with open('Data/pays_with_euro.pickle', "rb") as f:
    pays_with_euro = pickle.load(f)

#let's exchange currencies
#Check the format of the dates in dataframe
currency_exchange_file_path = r"Data\exchange_rates_ECB.csv"
df_currency_exchange = pd.read_csv(filepath_or_buffer=currency_exchange_file_path)

In [ ]:
def contains_number(text):
    text = str(text)
    pattern = r'\d+(?:[,.]\d+)?'
    numbers = re.findall(pattern, text)
    if len(numbers) > 0:
        return True
    else:
        return False

def get_columns_with_multiple(df):
    column_groups = []
    columns_with_multiple = []
    for col in [col for col in df.columns if contains_number(col) == True]:
        column_groups.append(col.split(":")[0])
        columns_with_multiple.append(col)

    column_groups = list(set(column_groups))

    return column_groups, columns_with_multiple

#drop all rows which do not match a certain condition
def impute_or_drop_rows(df, col, impute_or_drop, replacement_value = "standard value", condition = False):
    count = 0
    drop_rows_list = []
    if col not in df.columns:
        print("column not in df")

    if impute_or_drop != "drop" and impute_or_drop != "impute":
        print("invalid argument for impute or drop was provided")

    if replacement_value == "standard value" and impute_or_drop == "impute":
        print("imputation was the goal, but no alternative value was imputation was provided")

    for i in tqdm(range(0, len(df))):
        if condition == "nan":
            if pd.isna(df[col].iloc[i]) == True:
                if impute_or_drop == "impute":
                    df.at[i, col] = replacement_value
                    count += 1
                elif impute_or_drop == "drop":
                    drop_rows_list.append(i)
            else:
                continue
        else:
            if df[col].iloc[i] == condition:
                if impute_or_drop == "impute":
                    df.at[i, col] = replacement_value
                    count += 1
                elif impute_or_drop == "drop":
                    drop_rows_list.append(i)

    if impute_or_drop == "drop":
        print("A total of {} rows of the {} were dropped".format(len(drop_rows_list), len(df)))
        df = df.drop(labels = drop_rows_list, axis=0).reset_index(drop=True)
    elif impute_or_drop == "impute":
        print("A total of {} values of the {} were imputed or replaced".format(count, len(df)))
    
    return df

#Let's the columns for ca_other with ca_type with ca_type_other and ca_activity with ca_activity_other
def fuze_columns(df, main_col, supplemental_col):
    count = 0
    values = []
    for i in tqdm(range(0, len(df))):
        main_value = df[main_col].iloc[i]
        supplemental_value = df[supplemental_col].iloc[i]
        if pd.isna(main_value) == True and pd.isna(supplemental_value) == False:
            values.append(supplemental_value)
            count += 1
        elif pd.isna(main_value) == False:
            values.append(main_value)
        else:
            values.append(np.nan)
    df[main_col] = values
    df = df.drop(columns = supplemental_col)
    return print("{} cases were added to the main column {}".format(count, main_col))

def preprocess_weight_columns(text: str):
    text = str(text)
    pattern = r'\d+(?:[,.]\d+)?'
    numbers_list = re.findall(pattern, text)
    if len(numbers_list) > 0:
        for value in numbers_list:
            #the value has a comma, replace and multiply
            if "," in value:
                value = value.replace(",", ".")
                value = int(float(value) * 100)
            elif "." in value and int(float(value)) < 1:
                value = int(float(value)) * 100
            elif "." in value and int(float(value)) > 1:
                value = int(float(value))
            else:
                value = int(float(value))
    else:
        value = np.nan

    if value > 100:
        value = np.nan

    return value

def convert_to_days(df, value, i):
    average_days_per_month = 30.437
    if df["duration type"].iloc[i] == "MONTH":
        amount_in_days = int(int(value) * average_days_per_month)
    elif df["duration type"].iloc[i] == "DAY":
        amount_in_days = int(value)

    return amount_in_days

def get_numeric_data_multiple(df, features: list, specification = "not average", purpose = "produce statistics"):
    #create a list of columns on which we can aggregate
    columns_need_preprocessing = ['ac_price_weighting', 'ac_weighting', 'ac_cost/ac_weighting', "ac_price"]
    feature_columns_values = {}
    for feature in features:
        #get all columns matching the group identifier
        feature_columns = []
        for col in df.columns:
            if feature in col and "@" not in col:
                feature_columns.append(col)
            else:
                continue
        feature_columns_values[feature] = feature_columns
    
    aggregated_values = {}

    for feature in feature_columns_values.keys():
        values_per_feature = []
        for i in tqdm(range(0, len(df))):
            total = 0
            count_non_empty_cols = 0
            for col in feature_columns_values[feature]:
                if pd.isna(df[col].iloc[i]) == False:
                    value = df[col].iloc[i]
                    if feature in columns_need_preprocessing:
                        value = preprocess_weight_columns(value)
                    elif feature == "object_descr/duration":
                        value = convert_to_days(df, value, i)
                    else:
                        value = int(float(value))
                    
                    if pd.isna(value) == False:
                        total += value
                        count_non_empty_cols += 1
                    else:
                        continue
                    
                elif pd.isna(df[col].iloc[i]) == True:
                    continue
                else:
                    continue
            
            if total == 0 and purpose == "adjust df":
                total = np.nan
                values_per_feature.append(total)
            elif total == 0 and purpose == "produce statistics":
                continue
            elif total != 0 and pd.isna(total) == False and specification != "average":
                values_per_feature.append(total)
            elif total != 0 and pd.isna(total) == False and specification == "average":
                total = int(float(total / count_non_empty_cols))
                values_per_feature.append(total)
            
        aggregated_values[feature] = values_per_feature

    return aggregated_values

def get_numbers(text):
    pattern = r'\d+(?:[,.]\d+)?'
    values_list = re.findall(pattern, text)
    return values_list

def get_numeric_single(df, columns):
    data_numeric_single = {}

    for col in tqdm(columns):
        col_values = []
        for i in range(0, len(df)):
            if pd.isna(df[col].iloc[i]) == False:
                values = str(df[col].iloc[i])
                values_list = get_numbers(values)
                if len(values_list) > 0:
                    total = 0
                    for value in values_list:
                        total += int(float(value))
                    col_values.append(total)
                else:
                    col_values.append(np.nan)
            else:
                col_values.append(np.nan)
        data_numeric_single[col] = col_values
    return data_numeric_single

#let's work on the duration columns
#transform all date_start and date_end to datetime objects 
def merge_duration(df, main_col, fuze_cols):
    duration_column_calculated = []
    strange_days_count = []
    time_indices = []
    
    for index in range(0, len(df)):
        duration = df[main_col][index]
        start = df[fuze_cols[0]][index]
        end = df[fuze_cols[1]][index]
        
        if pd.isna(duration) != True and pd.isna(start) == True and pd.isna(end) == True:
            duration_column_calculated.append(duration)
            time_indices.append(index)

        elif pd.isna(duration) == True and pd.isna(start) != True and pd.isna(end) != True:
            start_date = datetime.strptime(str(start), "%Y-%m-%d")
            end_date = datetime.strptime(str(end),"%Y-%m-%d")
            delta_days = (end_date - start_date).days
            duration_column_calculated.append(int(delta_days))
            time_indices.append(index)

        elif pd.isna(duration) != True and (pd.isna(start) != True or pd.isna(end) != True):
            duration_column_calculated.append(duration)
            time_indices.append(index)

        else:
            duration_column_calculated.append(np.nan)
            strange_days_count.append(index)

    df["duration"] = duration_column_calculated
    return df

def check_currency(df, money_row_curs: list, money_list: dict, index_money_row: int, money_row: dict, money_key, currency_key, country_to_currency_mapping = country_to_currency_mapping):
    
    country = df["country"].iloc[index_money_row]
    some_curs = [currency for currency in money_row_curs if currency != None and len(currency) == 3]
    if len(some_curs) > 0:
        for k in range(0, len(some_curs)):
            if some_curs[0] != some_curs[len(some_curs)-1-k]:
                #they differ, I dont know what to do with it so set the money amount to None
                money_list[index_money_row][money_key] = None
                money_list[index_money_row][currency_key] = None
            else:
                #they do not differ, assign the currency that was found to be similar in the whole row
                assigned_currency = some_curs[0]
                money_list[index_money_row][currency_key] = assigned_currency
    elif df['country'].iloc[money_row["index"]] in [country for country in pays_with_euro.keys() if pays_with_euro[country] == 1]:
        #all currencies were none check if it is an euro_zone country
        money_list[index_money_row][currency_key] = 'EUR'
    
    elif country in country_to_currency_mapping.keys():
        assigned_currency = country_to_currency_mapping[country]
        money_list[index_money_row][currency_key] = assigned_currency
    else:
        return index_money_row
    

def exchange_currency(money_list: list, index_money_row: int, money_key: int, df_currency_exchange, money_row: dict, currency: str, year: str):
    if currency != None:
        currency_exchange_year = {2020: 0, 2021: 1}
        conversion_rate = df_currency_exchange[currency][currency_exchange_year[year]]
        total_amount = 0

        for money_piece in str(money_row[money_key]).split(" "):
            if len(re.findall("([\d.]*\d+)", money_piece)) > 0:
                money_piece = money_piece.split(".")[0] #just keep the whole number rather than what is after the comma
                total_amount += int(float(re.findall("([\d.]*\d+)", money_piece)[0]))
        
        if np.isnan(total_amount) == False:
            total_new_amount = int(total_amount / conversion_rate)
            money_list[index_money_row][money_key] = total_new_amount

def df_get_money(df, money_columns, currency_columns):
    rows_with_money = []
    df[currency_columns] = df[currency_columns].replace(np.nan, None)


    for i in range(0, len(df)):
        money_row = {}
        for col in money_columns:
            currency_col = col.split(":")[0] + "[@currency]"
            if pd.isna(df[col][i]) == False:
                money_row["index"] = i
                money_row[col] = df[col][i]
                money_row[currency_col] = df[currency_col][i]
        
        if len(money_row) > 0:
            rows_with_money.append(money_row)

    rows_no_currencies = []

    for index_money_row, money_row in tqdm(enumerate(rows_with_money)):
        money_row_currencies = [money_row[column_name] for column_name in money_row.keys() if "@" in column_name]

        for index_currency, currency in enumerate(money_row_currencies):
            index = "BS"
            money_key = list(money_row.keys())[index_currency*2+1]
            currency_key = list(money_row.keys())[index_currency*2+2]
        if currency == None:
            index = check_currency(df = df, money_row_curs=money_row_currencies, money_list=rows_with_money, index_money_row=index_money_row, money_row = money_row, money_key = money_key, currency_key=currency_key)
            exchange_currency(money_list=rows_with_money, index_money_row=index_money_row, df_currency_exchange=df_currency_exchange, money_row=money_row, currency=currency, money_key=money_key, year = 2020)
            if index != None:
                rows_no_currencies.append(rows_with_money[index]["index"])
            elif currency != None and currency in df_currency_exchange.columns:
                exchange_currency(money_list=rows_with_money, index_money_row=index_money_row, df_currency_exchange=df_currency_exchange, money_row=money_row, currency=currency, money_key=money_key, year = 2021)
            elif currency != None and currency not in df_currency_exchange.columns:
                #currency unknown, set value to nan
                rows_with_money[index_money_row][money_key] = np.nan

    rows_no_currencies = list(OrderedSet(rows_no_currencies))

    for row in rows_with_money:
        columns = list(row.keys())[1:]
        index = list(row.keys())[0]
        for col in columns:
            df.at[index, col] = row[col]
    
    df = df.fillna(value=np.nan)
    return df

def get_columns(df, features):
    feature_columns_values = {}
    for feature in features:
        feature_columns = []
        for col in df.columns:
            if feature in col.split(":")[0]:
                feature_columns.append(col)
            else:
                continue
        feature_columns_values[feature] = feature_columns
    return feature_columns_values

#aggregate all text columns of a given feature
def get_text_multiple(df, features, purpose = "produce statistics"):
    feature_columns_values_text = get_columns(df, features)
    aggregated_values_text = {}

    for feature in feature_columns_values_text.keys():
        values_per_feature = []
        for i in tqdm(range(0, len(df))):
            text_row = ""
            for col in feature_columns_values_text[feature]:
                if pd.isna(df[col].iloc[i]) == False:
                    text_row += (" " +str(df[col].iloc[i]))

            if len(text_row) == 0 and purpose == "produce statistics":
                continue
            elif len(text_row) == 0 and purpose == "adjust df":
                values_per_feature.append(np.nan)
            elif len(text_row) > 0:
                values_per_feature.append(text_row)
            else:
                print("strange case this is")
        
        aggregated_values_text[feature] = values_per_feature
    
    return aggregated_values_text

def get_text_single(df, columns, purpose):
    dict_of_text_values = {}
    for col in columns:
        text_per_col = []
        for i in range(0, len(df)):
            if pd.isna(df[col].iloc) == False:
                cell_text = str(df[col].iloc[i])
                text_per_col.append(cell_text)
            elif pd.isna(df[col].iloc) == True and purpose == "adjust df":
                text_per_col.append(np.nan)
            
        dict_of_text_values[col] = text_per_col
    return dict_of_text_values

#Let's the columns for ca_other with ca_type with ca_type_other and ca_activity with ca_activity_other
def fuze_target_cols(df, main_col, fuze_cols):
    low = fuze_cols[0]
    high = fuze_cols[1]
    values = []
    indices_no_target = []
    for i in range(0, len(df)):
        if pd.isna(df[low].iloc[i]) == False and pd.isna(df[high].iloc[i]) == False and pd.isna(df[main_col].iloc[i]) == True:
            values.append(df[low].iloc[i])
        elif pd.isna(df[low].iloc[i]) == True and pd.isna(df[high].iloc[i]) == False and pd.isna(df[main_col].iloc[i]) == True:
            values.append(df[high].iloc[i])
        elif pd.isna(df[low].iloc[i]) == False and pd.isna(df[high].iloc[i]) == True and pd.isna(df[main_col].iloc[i]) == True:
            values.append(df[low].iloc[i])
        elif pd.isna(df[low].iloc[i]) == True and pd.isna(df[high].iloc[i]) == True and pd.isna(df[main_col].iloc[i]) == False:
            values.append(df[main_col].iloc[i])
        elif pd.isna(df[low].iloc[i]) == False and pd.isna(df[high].iloc[i]) == False and pd.isna(df[main_col].iloc[i]) == False:
            values.append(df[main_col].iloc[i])
        elif pd.isna(df[low].iloc[i]) == True and pd.isna(df[high].iloc[i]) == True and pd.isna(df[main_col].iloc[i]) == True:
            indices_no_target.append(i)
        else:
            continue
    
    print("a total of {} rows of the {} were dropped because they have no target variable".format(len(indices_no_target), len(df)))
    df = df.drop(columns = fuze_cols)
    df = df.drop(labels = indices_no_target, axis = 0).reset_index(drop=True)
    df[main_col] = values

    return df

#transform cpv column to 2 letter acronyms
def transform_cpv(df, classification_type = 2):
    values = []
    for i in range(0, len(df)):
        value = df["cpv_code"].iloc[i]
        if pd.isna(value) == False:
            values.append(str(value)[:classification_type])
        else:
            values.append(np.nan)
    df["cpv"] = values
    return df

In [ ]:
#let's exchange currencies
#Check the format of the dates in dataframe
currency_exchange_file_path = r"Data\exchange_rates_ECB.csv"
df_currency_exchange = pd.read_csv(filepath_or_buffer=currency_exchange_file_path)

In [ ]:
currency_columns = [column for column in df.columns if "currency" in column]
rename_currency_columns = [col.split(".")[2] if len(col.split(".")) > 1 else col for col in df_currency_exchange.columns] #get the currency acronyms like 'EUR'
df_currency_exchange.columns = rename_currency_columns #rename the columns
df_currency_exchange = df_currency_exchange.drop(labels = [i for i in range(0, len(df_currency_exchange)) if i != 21 and i != 22], axis = 'index').reset_index() #drop exchange rates of irrelevant years and reset index
df_currency_exchange = df_currency_exchange.drop(labels = [col for col in df_currency_exchange.columns if df_currency_exchange[col].isnull().any()], axis = 1) #drop columns if they contain NaN

#dropping GIP, ERN, COP, AFN, OMR, GMD, ETB, FKP, PAB, RSD, USN, CYP, EEK, EGP, LTL, MTL
missing_currencies = {"MKD": [61.6404, 61.602], 
                      "MDL": [19.676, 20.8748],
                      "AMD": [555.555, 597.79],
                      "ALL": [123.7864, 122.4619],
                      "FJD": [2.4469, 2.3048],
                      "XOF": [656.0955, 655.9288],
                      "BDD": [2.2849 , 2.3676],
                      "MMK": [1570.00, 1905.74],
                      "SKK": [10.491, 10.1485],
                      "EUR": [1, 1]
                      }

for column in missing_currencies.keys():
    df_currency_exchange[column] = missing_currencies[column]

NaN_currency_columns = [currency for currency in df_currency_exchange.columns if df_currency_exchange[currency].isnull().any()]

In [ ]:
df_sample = copy.deepcopy(df[:10000])

In [ ]:
money_columns = [col for col in df_sample.columns if "val" in col and len(col.split("[")) < 2]
df_sample = df_get_money(df_sample, money_columns, currency_columns=currency_columns)

In [ ]:
#get all different column groups
categorical_columns = ["country", "ca_type", "ca_activity", "cpv_code", "type_contract", "duration type", "ca_type_other", "ca_activity_other", "renewal", "procedure", "recurrent", "joint_procurement_involved", "central_purchasing", "eu_fund",
                       "awarded_contract", "framework or dynamic purchasing"]
numerical_columns = ["object_contract/val_estimated_total", "object_contract/val_total", "award_contract/val_estimated_total", "award_contract/val_total", "object_descr/duration",
                     "ac_price_weighting", "ac_weighting", "ac_cost/ac_weighting", "nb_tenders_received", "nb_tenders_received_sme", "lowest offer", "highest offer", "ac_price"]

column_groups, columns_with_multiple = get_columns_with_multiple(df)
single_columns = [col for col in df.columns if col not in columns_with_multiple]
numerical_cols_single = [col for col in numerical_columns if col not in column_groups]
numerical_cols_multiple = [col for col in numerical_columns if col in column_groups]

In [ ]:
#drop all nan values for the col awarded_contract
df_sample = impute_or_drop_rows(df_sample, col = "awarded_contract", impute_or_drop="drop", condition = "nan")
#drop all values for the col awarded_contract if value == NO_AWARDED_CONTRACT
df_sample = impute_or_drop_rows(df_sample, col = "awarded_contract", impute_or_drop="drop", condition = "NO_AWARDED_CONTRACT")

#let's replace values for the categorical columns
df_sample = impute_or_drop_rows(df_sample, col="central_purchasing", impute_or_drop="impute", condition = "nan")
df_sample = impute_or_drop_rows(df_sample, col="joint_procurement_involved", impute_or_drop="impute", condition = "nan")
df_sample = impute_or_drop_rows(df_sample, col="framework or dynamic purchasing", impute_or_drop="impute", condition = "nan")

#let's merge the categorical columns
fuze_columns(df_sample, "ca_type", "ca_type_other")
fuze_columns(df_sample, "ca_activity", "ca_activity_other")

#let's merge the duration columns 
merge_duration(df_sample, main_col = "object_descr/duration: 0", fuze_cols = ["object_descr/date_start: 0", "object_descr/date_end: 0"])

In [ ]:
df_sample = df_sample.drop(labels = [9900], axis = 0)

In [ ]:
#split columns depending on averaging or adding values
numerical_cols_multiple_money = numerical_cols_multiple[:3]
numerical_cols_multiple_weighted = numerical_cols_multiple[3:]

numerical_cols_multiple_money_arrays = get_numeric_data_multiple(df_sample, numerical_cols_multiple_money, purpose="adjust df")
numerical_cols_multiple_weighted_arrays = get_numeric_data_multiple(df_sample, numerical_cols_multiple_weighted, "average", purpose="adjust df")
data_numeric_single = get_numeric_single(df_sample, numerical_cols_single)
aggregated_values_numeric = numerical_cols_multiple_money_arrays | numerical_cols_multiple_weighted_arrays | data_numeric_single

In [ ]:
#define the text columns
text_features = ["object_descr/title", "ac_cost/ac_criterion", "ac_criterion"]
text_columns_single = ["short_descr", "object_contract/title"]
text_columns_with_multiple = [col for col in columns_with_multiple if col.split(":")[0] in text_features]
text_columns = text_columns_with_multiple + text_columns_single

#!!UNCOMMENT TO RUN THE CODE AGAIN!!
aggregated_values_text_single = get_text_single(df_sample, text_columns_single, purpose = "adjust df")
aggregated_values_text_multiple = get_text_multiple(df_sample, text_features, purpose = "adjust df")
aggregated_values_text = aggregated_values_text_multiple | aggregated_values_text_single

In [ ]:
#alter df with our new values
aggregated_values_text_num = aggregated_values_text | aggregated_values_numeric

for feature in aggregated_values_text_num:
    df_sample[feature] = aggregated_values_text_num[feature]

In [ ]:
#numerical columns have been cleaned, fuze lowest and highest offer with award_contract/val_total
fuze_cols = ["lowest offer", "highest offer"]
df_sample = fuze_target_cols(df_sample, "award_contract/val_total", fuze_cols)

#preprocess the cpv codes
df_sample = transform_cpv(df_sample)

In [ ]:
df_sample = df_sample.drop(columns = columns_with_multiple)

In [ ]:
df_sample.head()